# 3. Open-platform query with a REQUIRE clause

Open platforms (`Platform$BDC_OPEN`, `Platform$BDC_DEV_OPEN`, `Platform$NHANES_OPEN`) expose study-level cross-counts without authentication. No token is needed — just pass the platform.

This notebook builds a `REQUIRE` clause — which matches participants for whom the concept has any non-null value — and runs it as a `CROSS_COUNT` to get counts broken down by consent group.

In [ ]:
library(picsure)

In [ ]:
open_session <- picsure::connect(
  platform = picsure::Platform$BDC_OPEN
)

## Find a study's CONSENT variable

In [ ]:
study_facet <- picsure::facets(open_session)
picsure::addFacet(study_facet, "dataset_id", "phs004002")

In [ ]:
study_consents <- picsure::searchDictionary(
  open_session,
  "CONSENT",
  facets = study_facet
)

In [ ]:
study_consents$conceptPath[[1]]

In [ ]:
study_consents$values[[1]]

## Build a REQUIRE clause

`REQUIRE` matches any participant with a non-null value for the concept. No `categories`/`min`/`max` needed.

In [ ]:
consent_clause <- picsure::buildClause(
  study_consents$conceptPath[[1]],
  type = picsure::PhenotypicFilterType$REQUIRE
)
consent_clause

## CROSS_COUNT

`CROSS_COUNT` returns a list of `CountResult` objects keyed by concept path — useful for breaking a single REQUIRE clause down across its category values.

In [ ]:
consent_query_result <- picsure::runQuery(
  open_session,
  consent_clause,
  type = picsure::QueryType$CROSS_COUNT
)
consent_query_result